# Этап 11 V1 — RealMLP против GBDT_mean

## Исследовательский вопрос

Может ли преднастроенный RealMLP на тех же 47 разрешённых признаках дать существенное улучшение относительно принятого B*=GBDT_mean?

Проверяется ровно одна модель: официальный PyTabKit RealMLP_TD_Classifier. Это официальная преднастроенная конфигурация, а не HPO, не ensemble и не AutoGluon wrapper.

Неизменны dataset Data_final.xlsb, target DefMark, identifier INN, рабочая выборка из 289614 строк, порядок строк, 47 признаков, outer StratifiedKFold(3, shuffle=True, random_state=42), seeds 43/44/45 и final test. Q_B1_norm и Q_B2_norm не являются predictors. GBDT заново не обучается: baseline берётся из сохранённого leakage-safe Stage 7 OOF.

Параметры, фиксируемые для контролируемого outer-fold protocol: device=cpu, n_cv=1, n_refit=0, n_ens=1 и fold-specific random_state. Никаких HPO, bagging/ensembling, calibration, class weighting, balancing, sampling или threshold optimization нет. Встроенная предобработка и внутренняя validation/early stopping RealMLP обучаются только на соответствующем outer-train fold.

Основная метрика выбора — полный OOF Gini. Precision, Recall и F1 при 0.5 являются только диагностикой. Правило решения зафиксировано до запуска: material_gain, если ΔGini >= +0.010 и RealMLP выигрывает Gini минимум на 2/3 folds; inferior, если ΔGini <= -0.010 и проигрывает минимум на 2/3; иначе no_material_benefit.


### Что проверяем?

Подготавливаем воспроизводимый contract: пути, feature identity, hashes, CV и выходные artifacts. Это нужно сделать до чтения данных и до обучения, чтобы experiment не мог молча изменить locked design. Неизменными остаются все Stage 1–10 artifacts.


In [1]:
from __future__ import annotations

import copy
import hashlib
import importlib.metadata
import json
import os
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
from pytabkit import RealMLP_TD_Classifier
from pytabkit.models.sklearn.default_params import DefaultParams
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent
GENERATED, SUMMARY = ROOT / 'reports' / 'generated', ROOT / 'reports' / 'summary'
DATASET = ROOT / 'data' / 'raw' / 'Data_final.xlsb'
STAGE1_PATH = GENERATED / 'stage1_baseline_results_V2.json'
STAGE7_PATH = GENERATED / 'stage7_tabm_stacking_results_V1.json'
STAGE7_OOF_PATH = GENERATED / 'stage7_tabm_stacking_oof_V1.npz'
RESULT_PATH = GENERATED / 'stage11_realmlp_results_V1.json'
OOF_PATH = GENERATED / 'stage11_realmlp_oof_V1.npz'
SUMMARY_PATH = SUMMARY / 'stage11_realmlp_summary_V1.json'

TARGET, IDENTIFIER = 'DefMark', 'INN'
FORBIDDEN = ('Q_B1_norm', 'Q_B2_norm')
EXPECTED_DATASET_SHA = 'fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930'
EXPECTED_WORKING_SHA = '80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45'
EXPECTED_N, OUTER_SEED, FOLD_SEEDS, THRESHOLD = 289614, 42, (43, 44, 45), 0.5
EXPERIMENT_OVERRIDES = {'device': 'cpu', 'n_cv': 1, 'n_refit': 0, 'n_ens': 1, 'verbosity': 2}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def sha256_indices(indices: np.ndarray) -> str:
    return hashlib.sha256(np.asarray(indices, dtype=np.int64).tobytes()).hexdigest()

def metrics_at_threshold(target: np.ndarray, probability: np.ndarray) -> dict[str, float]:
    predicted = (probability >= THRESHOLD).astype(np.int8)
    auc = float(roc_auc_score(target, probability))
    return {'ROC-AUC': auc, 'Gini': 2.0 * auc - 1.0, 'PR-AUC': float(average_precision_score(target, probability)), 'Precision': float(precision_score(target, predicted, zero_division=0)), 'Recall': float(recall_score(target, predicted, zero_division=0)), 'F1': float(f1_score(target, predicted, zero_division=0))}

def json_safe(value):
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return repr(value)

OFFICIAL_REALMLP_TD_CLASS_DEFAULTS = json_safe(copy.deepcopy(DefaultParams().RealMLP_TD_CLASS))

def effective_realmlp_config(seed: int) -> dict:
    fold_overrides = {**EXPERIMENT_OVERRIDES, 'random_state': seed}
    return {**copy.deepcopy(OFFICIAL_REALMLP_TD_CLASS_DEFAULTS), **fold_overrides}

def atomic_json(path: Path, payload: dict) -> None:
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False, suffix='.tmp') as stream:
        json.dump(payload, stream, ensure_ascii=False, indent=2, allow_nan=False)
    os.replace(stream.name, path)

def atomic_npz(path: Path, **arrays: np.ndarray) -> None:
    with tempfile.NamedTemporaryFile('wb', dir=path.parent, delete=False, suffix='.tmp') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(stream.name, path)

print(f'PyTabKit: {importlib.metadata.version("pytabkit")}; устройство: cpu')
print(f'Выходные artifacts: {RESULT_PATH.name}, {OOF_PATH.name}, {SUMMARY_PATH.name}')
print(f'Официальная конфигурация RealMLP_TD_CLASS: {json.dumps(OFFICIAL_REALMLP_TD_CLASS_DEFAULTS, ensure_ascii=False, sort_keys=True)}')


PyTabKit: 1.7.3; устройство: cpu
Выходные artifacts: stage11_realmlp_results_V1.json, stage11_realmlp_oof_V1.npz, stage11_realmlp_summary_V1.json
Официальная конфигурация RealMLP_TD_CLASS: {"act": "selu", "act_lr_factor": 0.1, "add_front_scale": true, "bias_init_mode": "he+5", "bias_lr_factor": 0.1, "bias_wd_factor": 0.0, "block_str": "w-b-a-d", "embedding_size": 8, "hidden_sizes": [256, 256, 256], "lr": 0.04, "lr_sched": "coslog4", "ls_eps": 0.1, "max_one_hot_cat_size": 9, "n_epochs": 256, "num_emb_type": "pbld", "opt": "adam", "p_drop": 0.15, "p_drop_sched": "flat_cos", "plr_hidden_1": 16, "plr_hidden_2": 4, "plr_lr_factor": 0.1, "plr_sigma": 0.1, "scale_lr_factor": 6.0, "sq_mom": 0.95, "tfms": ["one_hot", "median_center", "robust_scale", "smooth_clip", "embedding"], "use_ls": true, "use_parametric_act": true, "wd": 0.02, "wd_sched": "flat_cos", "weight_init_mode": "std", "weight_param": "ntk"}


### Что проверяем?

Проверяем preflight до обучения: идентичность dataset и working indices, target/fold alignment, baseline GBDT_mean и набор ровно из 47 разрешённых features. Это непосредственно исключает leakage через validation fold и останавливает обучение при несоответствии принятому control. Никакая preprocessing information из outer-validation здесь не создаётся.


In [2]:
stage1 = json.loads(STAGE1_PATH.read_text(encoding='utf-8'))
stage7 = json.loads(STAGE7_PATH.read_text(encoding='utf-8'))
assert DATASET.exists() and STAGE7_OOF_PATH.exists(), 'STOP: отсутствует обязательный input artifact'
assert sha256_file(DATASET) == EXPECTED_DATASET_SHA, 'STOP: SHA dataset не совпадает'
features = list(stage1['допустимые_признаки'])
assert len(features) == 47 and len(set(features)) == 47, 'STOP: число или уникальность features не совпадают'
assert not set(FORBIDDEN).intersection(features), 'STOP: найден запрещённый predictor'
assert features == stage7['raw_features_in_order'], 'STOP: feature identity Stage 1/7 не совпадает'
assert stage7['baseline_selection']['B_star'] == 'GBDT_mean', 'STOP: принятый B* не совпадает'
assert stage7['dataset_sha256'] == EXPECTED_DATASET_SHA, 'STOP: SHA dataset Stage 7 не совпадает'
assert stage7['working_index_sha256'] == EXPECTED_WORKING_SHA, 'STOP: SHA working-index Stage 7 не совпадает'

raw = pd.read_excel(DATASET, engine='pyxlsb')
assert TARGET in raw and IDENTIFIER in raw, 'STOP: target или identifier отсутствует'
assert all(column in raw for column in features), 'STOP: отсутствует обязательный feature'
with np.load(STAGE7_OOF_PATH, allow_pickle=False) as artifact:
    required = {'working_indices', 'target', 'fold', 'gbdt_mean'}
    assert required.issubset(artifact.files), f'STOP: отсутствуют OOF keys: {required - set(artifact.files)}'
    working_indices = np.asarray(artifact['working_indices'], dtype=np.int64)
    y_working = np.asarray(artifact['target'], dtype=np.int8)
    fold = np.asarray(artifact['fold'], dtype=np.int8)
    gbdt_mean = np.asarray(artifact['gbdt_mean'], dtype=np.float64)

assert len(working_indices) == len(y_working) == len(fold) == len(gbdt_mean) == EXPECTED_N, 'STOP: неожиданное working n'
assert np.unique(working_indices).size == EXPECTED_N, 'STOP: дублируются working indices'
assert sha256_indices(working_indices) == EXPECTED_WORKING_SHA, 'STOP: SHA working-index не совпадает'
assert np.array_equal(raw.loc[working_indices, TARGET].to_numpy(dtype=np.int8), y_working), 'STOP: target alignment не совпадает'
assert np.isfinite(gbdt_mean).all() and ((0.0 <= gbdt_mean) & (gbdt_mean <= 1.0)).all(), 'STOP: GBDT_mean содержит невалидные значения'
assert set(np.unique(fold)) == {1, 2, 3}, 'STOP: невалидные fold labels'

splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=OUTER_SEED)
expected_fold = np.zeros(EXPECTED_N, dtype=np.int8)
for fold_id, (_, valid_pos) in enumerate(splitter.split(np.zeros(EXPECTED_N), y_working), start=1):
    expected_fold[valid_pos] = fold_id
assert np.array_equal(fold, expected_fold), 'STOP: сохранённое fold assignment отличается от locked outer CV'

X_working = raw.loc[working_indices, features].copy()
assert not X_working.isna().any().any(), 'STOP: RealMLP не принимает пропуски в numeric features; imputation здесь запрещён'
assert all(pd.api.types.is_numeric_dtype(X_working[column]) for column in features), 'STOP: обнаружен нечисловой feature'
assert np.isfinite(X_working.to_numpy(dtype=np.float64)).all(), 'STOP: feature содержит неfinite значение'
print(f'ПРЕДВАРИТЕЛЬНАЯ ПРОВЕРКА ПРОЙДЕНА: n={EXPECTED_N}, features={len(features)}, baseline finite, dataset/indices/target/folds согласованы.')


ПРЕДВАРИТЕЛЬНАЯ ПРОВЕРКА ПРОЙДЕНА: n=289614, features=47, baseline finite, dataset/indices/target/folds согласованы.


### Что проверяем?

Выполняем ровно три outer folds. Для каждого RealMLP получает только X_train/y_train; его официальная встроенная предобработка и внутренняя validation остаются fold-local. В notebook выводятся индикаторы по fold, стадии и elapsed time, а параметры конструктора сохраняются отдельно от эффективной преднастроенной конфигурации в result JSON. Это отвечает на вопрос сравнением свежего RealMLP OOF с уже сохранённым GBDT_mean без повторного GBDT run.

Полный запуск дорогой и запускается пользователем только после pre-run review.


In [3]:
import contextlib
import threading
from IPython.display import HTML, display

CHECKPOINT_PATH = GENERATED / 'stage11_realmlp_checkpoint_V1.npz'

_STAGE11_PANEL = None
_STAGE11_PANEL_LOCK = threading.Lock()
_STAGE11_PROGRESS_LOCK = threading.Lock()

_STAGE11_PROGRESS = {
    'fold_id': 1,
    'phase': 'подготовка',
    'fold_started': None,
    'last_delta_gini': None,
}


def format_runtime(seconds: float | None) -> str:
    """Человекочитаемое время для progress-panel."""
    if seconds is None:
        return '—'

    total = max(0, int(round(float(seconds))))
    hours, remainder = divmod(total, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours:
        return f'{hours} ч {minutes:02d} мин {secs:02d} сек'
    if minutes:
        return f'{minutes} мин {secs:02d} сек'
    return f'{secs} сек'


def checkpoint_contract() -> dict:
    """Поля, которые обязаны совпасть при resume."""
    return {
        'experiment': 'Stage 11',
        'version': 'V1',
        'dataset_sha256': EXPECTED_DATASET_SHA,
        'working_index_sha256': EXPECTED_WORKING_SHA,
        'raw_features_in_order': list(features),
        'outer_seed': OUTER_SEED,
        'fold_seeds': list(FOLD_SEEDS),
        'experiment_overrides': json_safe(EXPERIMENT_OVERRIDES),
        'pytabkit_version': importlib.metadata.version('pytabkit'),
    }


def fresh_stage11_state() -> dict:
    """Новое состояние Stage 11 без завершённых фолдов."""
    return {
        **checkpoint_contract(),
        'status': 'in_progress',
        'completed_folds': [],
        'fold_rows': [],
        'effective_configs': [],
        'runtime_seconds': 0.0,
        'active_fold': None,
        'phase': 'подготовка',
        'realmlp_oof': np.full(
            EXPECTED_N,
            np.nan,
            dtype=np.float64,
        ),
    }


def save_stage11_checkpoint(state: dict) -> None:
    """
    Атомарно сохраняем промежуточное состояние.

    Сам checkpoint не объявляет эксперимент completed.
    """
    metadata = {
        key: value
        for key, value in state.items()
        if key != 'realmlp_oof'
    }

    atomic_npz(
        CHECKPOINT_PATH,
        realmlp_oof=np.asarray(
            state['realmlp_oof'],
            dtype=np.float64,
        ),
        state_json=np.asarray(
            json.dumps(
                json_safe(metadata),
                ensure_ascii=False,
                allow_nan=False,
            )
        ),
    )


def load_stage11_checkpoint() -> tuple[dict, str]:
    """
    Загружаем checkpoint и проверяем его совместимость
    с текущим locked Stage 11.
    """
    if not CHECKPOINT_PATH.exists():
        state = fresh_stage11_state()
        save_stage11_checkpoint(state)
        return state, 'fresh'

    with np.load(
        CHECKPOINT_PATH,
        allow_pickle=False,
    ) as artifact:
        required = {
            'realmlp_oof',
            'state_json',
        }

        if not required.issubset(artifact.files):
            raise RuntimeError(
                'STOP: checkpoint Stage 11 имеет неожиданную структуру'
            )

        realmlp_oof = np.asarray(
            artifact['realmlp_oof'],
            dtype=np.float64,
        )

        metadata = json.loads(
            str(artifact['state_json'].item())
        )

    expected = checkpoint_contract()

    for key, expected_value in expected.items():
        if metadata.get(key) != expected_value:
            raise RuntimeError(
                f'STOP: checkpoint Stage 11 несовместим '
                f'по полю {key!r}'
            )

    if realmlp_oof.shape != (EXPECTED_N,):
        raise RuntimeError(
            'STOP: checkpoint Stage 11 содержит OOF '
            'неправильной длины'
        )

    completed_folds = {
        int(value)
        for value in metadata.get(
            'completed_folds',
            [],
        )
    }

    if not completed_folds.issubset({1, 2, 3}):
        raise RuntimeError(
            'STOP: checkpoint Stage 11 содержит неизвестный fold'
        )

    fold_rows = metadata.get(
        'fold_rows',
        [],
    )

    config_rows = metadata.get(
        'effective_configs',
        [],
    )

    row_folds = [
        int(row['fold'])
        for row in fold_rows
    ]

    config_folds = [
        int(row['fold'])
        for row in config_rows
    ]

    if (
        len(row_folds) != len(set(row_folds))
        or set(row_folds) != completed_folds
    ):
        raise RuntimeError(
            'STOP: checkpoint Stage 11 содержит '
            'несогласованные fold metrics'
        )

    if (
        len(config_folds) != len(set(config_folds))
        or set(config_folds) != completed_folds
    ):
        raise RuntimeError(
            'STOP: checkpoint Stage 11 содержит '
            'несогласованные configs'
        )

    for fold_id in (1, 2, 3):
        positions = fold == fold_id

        if fold_id in completed_folds:
            if not np.isfinite(
                realmlp_oof[positions]
            ).all():
                raise RuntimeError(
                    f'STOP: fold {fold_id} отмечен завершённым, '
                    'но его OOF заполнен не полностью'
                )

        else:
            # Незавершённый фолд никогда не используем частично.
            realmlp_oof[positions] = np.nan

    state = {
        **metadata,
        'completed_folds': sorted(
            completed_folds
        ),
        'realmlp_oof': realmlp_oof,
    }

    return state, 'resume'


def set_stage11_progress(**updates) -> None:
    """Безопасно обновляем состояние progress-panel."""
    with _STAGE11_PROGRESS_LOCK:
        _STAGE11_PROGRESS.update(updates)


def stage11_progress_snapshot() -> dict:
    with _STAGE11_PROGRESS_LOCK:
        return dict(_STAGE11_PROGRESS)


def render_stage11_panel(
    state: dict,
    session_started: float,
    base_runtime: float,
) -> None:
    """Одна обновляемая человекочитаемая progress-panel."""
    global _STAGE11_PANEL

    progress = stage11_progress_snapshot()
    now = time.monotonic()

    completed = {
        int(value)
        for value in state.get(
            'completed_folds',
            [],
        )
    }

    completed_count = len(completed)

    fold_started = progress.get(
        'fold_started'
    )

    current_fold_elapsed = (
        None
        if fold_started is None
        else now - float(fold_started)
    )

    total_elapsed = (
        base_runtime
        + (now - session_started)
    )

    fold_times = [
        float(row['runtime_seconds'])
        for row in state.get(
            'fold_rows',
            [],
        )
    ]

    average_fold = (
        float(np.mean(fold_times))
        if fold_times
        else None
    )

    if average_fold is None:
        eta_text = (
            'появится после первого завершённого фолда'
        )

    else:
        current_fold = int(
            progress['fold_id']
        )

        unfinished_current = 0.0

        if (
            current_fold not in completed
            and current_fold_elapsed is not None
        ):
            unfinished_current = max(
                average_fold
                - current_fold_elapsed,
                0.0,
            )

        future_folds = sum(
            1
            for fold_id in (1, 2, 3)
            if (
                fold_id not in completed
                and fold_id != current_fold
            )
        )

        eta_seconds = (
            unfinished_current
            + future_folds * average_fold
        )

        eta_text = (
            f'оценка: '
            f'{format_runtime(eta_seconds)}'
        )

    checkpoint_text = (
        f'после фолда {max(completed)}'
        if completed
        else 'создан; завершённых фолдов пока нет'
    )

    delta = progress.get(
        'last_delta_gini'
    )

    delta_text = (
        '—'
        if delta is None
        else f'{float(delta):+.6f}'
    )

    html = HTML(
        f"""
        <div style="
            font-family: Arial, sans-serif;
            border: 1px solid #bdbdbd;
            border-radius: 10px;
            padding: 14px 16px;
            max-width: 760px;
            line-height: 1.55;
        ">
            <h3 style="margin:0 0 12px 0;">
                Этап 11 V1 — RealMLP
            </h3>

            <table style="border-collapse:collapse;">
                <tr>
                    <td style="padding-right:28px;">
                        <b>Внешний фолд</b>
                    </td>
                    <td>{int(progress['fold_id'])}/3</td>
                </tr>

                <tr>
                    <td><b>Текущая стадия</b></td>
                    <td>{progress['phase']}</td>
                </tr>

                <tr>
                    <td><b>Завершено фолдов</b></td>
                    <td>{completed_count}/3</td>
                </tr>

                <tr>
                    <td><b>Время текущего фолда</b></td>
                    <td>{format_runtime(current_fold_elapsed)}</td>
                </tr>

                <tr>
                    <td><b>Общее время Stage 11</b></td>
                    <td>{format_runtime(total_elapsed)}</td>
                </tr>

                <tr>
                    <td><b>Среднее время завершённого фолда</b></td>
                    <td>{format_runtime(average_fold)}</td>
                </tr>

                <tr>
                    <td><b>Осталось</b></td>
                    <td>{eta_text}</td>
                </tr>

                <tr>
                    <td><b>Checkpoint</b></td>
                    <td>{checkpoint_text}</td>
                </tr>

                <tr>
                    <td><b>Последний ΔGini</b></td>
                    <td>{delta_text}</td>
                </tr>
            </table>

            <div style="
                margin-top:10px;
                font-size:12px;
            ">
                ETA — оценка по фактически завершённым фолдам.
                Если остановить незавершённый фолд,
                при resume он начнётся заново.
            </div>
        </div>
        """
    )

    with _STAGE11_PANEL_LOCK:
        if _STAGE11_PANEL is None:
            _STAGE11_PANEL = display(
                html,
                display_id=True,
            )
        else:
            _STAGE11_PANEL.update(html)


def stage11_heartbeat(
    stop_event: threading.Event,
    state: dict,
    session_started: float,
    base_runtime: float,
) -> None:
    """Обновляем elapsed time примерно каждые 30 секунд."""
    while not stop_event.wait(30.0):
        try:
            render_stage11_panel(
                state,
                session_started,
                base_runtime,
            )

        except Exception:
            # Ошибка только визуализации
            # не должна останавливать ML-run.
            pass


def fit_realmlp_quietly(
    model: RealMLP_TD_Classifier,
    x_train: pd.DataFrame,
    y_train: np.ndarray,
) -> None:
    """
    Запускаем тот же model.fit(), но не выводим в notebook
    технические логи PyTabKit и локальные пути.

    verbosity=2 в locked конфигурации НЕ меняется.
    """
    with tempfile.TemporaryFile(
        mode='w+',
        encoding='utf-8',
    ) as technical_output:

        with (
            contextlib.redirect_stdout(
                technical_output
            ),
            contextlib.redirect_stderr(
                technical_output
            ),
        ):
            model.fit(
                x_train,
                y_train,
            )


def show_stage11_result(
    result: dict,
) -> None:
    """Короткий человекочитаемый итог вместо сырого JSON."""
    realmlp_metrics = (
        result['realmlp_oof_metrics']
    )

    baseline_metrics = (
        result['gbdt_mean_oof_metrics']
    )

    deltas = (
        result[
            'delta_realmlp_minus_gbdt_mean'
        ]
    )

    decision_text = {
        'material_gain':
            'существенное улучшение',
        'inferior':
            'RealMLP хуже baseline',
        'no_material_benefit':
            'материального преимущества нет',
    }.get(
        result['decision'],
        result['decision'],
    )

    print('Stage 11 V1 завершён.')
    print(
        f"RealMLP Gini:   "
        f"{realmlp_metrics['Gini']:.6f}"
    )
    print(
        f"GBDT_mean Gini: "
        f"{baseline_metrics['Gini']:.6f}"
    )
    print(
        f"ΔGini:          "
        f"{deltas['Gini']:+.6f}"
    )
    print(
        f"Решение:        "
        f"{decision_text} "
        f"({result['decision']})"
    )
    print(
        f"Общее время:    "
        f"{format_runtime(result['runtime_seconds'])}"
    )


def decide(
    delta_gini: float,
    fold_deltas: list[float],
) -> str:
    """
    Исходное locked decision rule Stage 11.
    Логика не изменена.
    """
    wins = sum(
        delta > 0.0
        for delta in fold_deltas
    )

    losses = sum(
        delta < 0.0
        for delta in fold_deltas
    )

    if (
        delta_gini >= 0.010
        and wins >= 2
    ):
        return 'material_gain'

    if (
        delta_gini <= -0.010
        and losses >= 2
    ):
        return 'inferior'

    return 'no_material_benefit'


def run_stage11() -> dict:
    """
    Полный Stage 11 с fold-level checkpoint/resume.

    ML-протокол исходного эксперимента не меняется.
    """
    global _STAGE11_PANEL
    _STAGE11_PANEL = None

    # RESULT_PATH пишется последним.
    # Поэтому completed result защищает от повторного ML-run.
    if RESULT_PATH.exists():
        existing = json.loads(
            RESULT_PATH.read_text(
                encoding='utf-8'
            )
        )

        if existing.get('status') == 'completed':

            if (
                existing.get('dataset_sha256')
                != EXPECTED_DATASET_SHA
            ):
                raise RuntimeError(
                    'STOP: completed Stage 11 result '
                    'относится к другому dataset'
                )

            if (
                existing.get(
                    'working_index_sha256'
                )
                != EXPECTED_WORKING_SHA
            ):
                raise RuntimeError(
                    'STOP: completed Stage 11 result '
                    'относится к другой working sample'
                )

            if (
                not OOF_PATH.exists()
                or not SUMMARY_PATH.exists()
            ):
                raise RuntimeError(
                    'STOP: completed result найден, '
                    'но комплект Stage 11 artifacts неполный'
                )

            print(
                'Stage 11 уже завершён; '
                'повторный ML-run не запускается.'
            )

            show_stage11_result(existing)
            return existing

    state, launch_mode = (
        load_stage11_checkpoint()
    )

    session_started = (
        time.monotonic()
    )

    base_runtime = float(
        state.get(
            'runtime_seconds',
            0.0,
        )
    )

    finished = False

    set_stage11_progress(
        fold_id=1,
        phase=f'подготовка ({launch_mode})',
        fold_started=None,
        last_delta_gini=None,
    )

    render_stage11_panel(
        state,
        session_started,
        base_runtime,
    )

    stop_event = threading.Event()

    heartbeat = threading.Thread(
        target=stage11_heartbeat,
        args=(
            stop_event,
            state,
            session_started,
            base_runtime,
        ),
        daemon=True,
    )

    heartbeat.start()

    try:
        for fold_id, seed in enumerate(
            FOLD_SEEDS,
            start=1,
        ):
            completed = {
                int(value)
                for value
                in state['completed_folds']
            }

            if fold_id in completed:
                existing_row = next(
                    row
                    for row in state['fold_rows']
                    if int(row['fold']) == fold_id
                )

                set_stage11_progress(
                    fold_id=fold_id,
                    phase=(
                        'фолд уже завершён — '
                        'пропускаем'
                    ),
                    fold_started=None,
                    last_delta_gini=(
                        existing_row[
                            'delta_realmlp_minus_gbdt_mean'
                        ]['Gini']
                    ),
                )

                render_stage11_panel(
                    state,
                    session_started,
                    base_runtime,
                )

                continue

            train_pos = np.flatnonzero(
                fold != fold_id
            )

            valid_pos = np.flatnonzero(
                fold == fold_id
            )

            fold_started = (
                time.monotonic()
            )

            state['active_fold'] = fold_id
            state['phase'] = (
                'обучение RealMLP'
            )

            state['runtime_seconds'] = (
                base_runtime
                + (
                    time.monotonic()
                    - session_started
                )
            )

            # Checkpoint существует ещё до начала fit.
            save_stage11_checkpoint(state)

            set_stage11_progress(
                fold_id=fold_id,
                phase='обучение RealMLP',
                fold_started=fold_started,
                last_delta_gini=None,
            )

            render_stage11_panel(
                state,
                session_started,
                base_runtime,
            )

            model = RealMLP_TD_Classifier(
                random_state=seed,
                **EXPERIMENT_OVERRIDES,
            )

            config_record = {
                'fold': fold_id,
                'seed': seed,
                'official_tuned_default_recipe':
                    copy.deepcopy(
                        OFFICIAL_REALMLP_TD_CLASS_DEFAULTS
                    ),
                'experiment_overrides': {
                    **EXPERIMENT_OVERRIDES,
                    'random_state': seed,
                },
                'effective_realmlp_config':
                    effective_realmlp_config(seed),
                'constructor_params':
                    json_safe(
                        model.get_params(
                            deep=False
                        )
                    ),
            }

            fit_realmlp_quietly(
                model,
                X_working.iloc[
                    train_pos
                ],
                y_working[
                    train_pos
                ],
            )

            state['phase'] = (
                'прогноз outer-validation'
            )

            set_stage11_progress(
                phase='прогноз outer-validation'
            )

            render_stage11_panel(
                state,
                session_started,
                base_runtime,
            )

            probability = np.asarray(
                model.predict_proba(
                    X_working.iloc[
                        valid_pos
                    ]
                )[:, 1],
                dtype=np.float64,
            )

            assert (
                np.isfinite(
                    probability
                ).all()
                and (
                    (
                        0.0
                        <= probability
                    )
                    & (
                        probability
                        <= 1.0
                    )
                ).all()
            ), (
                'STOP: RealMLP probability '
                'невалидна'
            )

            model_metrics = (
                metrics_at_threshold(
                    y_working[
                        valid_pos
                    ],
                    probability,
                )
            )

            baseline_metrics = (
                metrics_at_threshold(
                    y_working[
                        valid_pos
                    ],
                    gbdt_mean[
                        valid_pos
                    ],
                )
            )

            delta = {
                key:
                    model_metrics[key]
                    - baseline_metrics[key]
                for key
                in model_metrics
            }

            # Только после prediction + metrics
            # этот fold считается завершённым.
            state[
                'realmlp_oof'
            ][valid_pos] = probability

            state[
                'fold_rows'
            ].append(
                {
                    'fold': fold_id,
                    'seed': seed,
                    'n_train':
                        int(
                            train_pos.size
                        ),
                    'n_validation':
                        int(
                            valid_pos.size
                        ),
                    'runtime_seconds':
                        (
                            time.monotonic()
                            - fold_started
                        ),
                    'realmlp_metrics':
                        model_metrics,
                    'gbdt_mean_metrics':
                        baseline_metrics,
                    'delta_realmlp_minus_gbdt_mean':
                        delta,
                }
            )

            state[
                'effective_configs'
            ].append(
                config_record
            )

            state['completed_folds'] = (
                sorted(
                    completed
                    | {fold_id}
                )
            )

            state['active_fold'] = None
            state['phase'] = (
                'checkpoint сохранён'
            )

            state['runtime_seconds'] = (
                base_runtime
                + (
                    time.monotonic()
                    - session_started
                )
            )

            # Главный resumable checkpoint:
            # полностью законченный outer fold.
            save_stage11_checkpoint(state)

            set_stage11_progress(
                fold_id=fold_id,
                phase=(
                    'фолд завершён; '
                    'checkpoint сохранён'
                ),
                fold_started=None,
                last_delta_gini=(
                    delta['Gini']
                ),
            )

            render_stage11_panel(
                state,
                session_started,
                base_runtime,
            )

        assert (
            set(
                int(value)
                for value
                in state[
                    'completed_folds'
                ]
            )
            == {1, 2, 3}
        ), (
            'STOP: не все outer folds '
            'завершены'
        )

        realmlp_oof = np.asarray(
            state['realmlp_oof'],
            dtype=np.float64,
        )

        assert np.isfinite(
            realmlp_oof
        ).all(), (
            'STOP: RealMLP OOF '
            'заполнен не полностью'
        )

        fold_rows = sorted(
            state['fold_rows'],
            key=lambda row:
                int(row['fold']),
        )

        effective_configs = sorted(
            state[
                'effective_configs'
            ],
            key=lambda row:
                int(row['fold']),
        )

        realmlp_metrics = (
            metrics_at_threshold(
                y_working,
                realmlp_oof,
            )
        )

        baseline_metrics = (
            metrics_at_threshold(
                y_working,
                gbdt_mean,
            )
        )

        deltas = {
            key:
                realmlp_metrics[key]
                - baseline_metrics[key]
            for key
            in realmlp_metrics
        }

        fold_gini_deltas = [
            row[
                'delta_realmlp_minus_gbdt_mean'
            ]['Gini']
            for row
            in fold_rows
        ]

        total_runtime = (
            base_runtime
            + (
                time.monotonic()
                - session_started
            )
        )

        result = {
            'experiment': 'Stage 11',
            'version': 'V1',
            'status': 'completed',
            'dataset_sha256':
                EXPECTED_DATASET_SHA,
            'working_index_sha256':
                EXPECTED_WORKING_SHA,
            'target': TARGET,
            'identifier': IDENTIFIER,
            'raw_features_in_order':
                features,
            'raw_feature_count':
                len(features),
            'forbidden_features':
                list(FORBIDDEN),
            'pytabkit_version':
                importlib.metadata.version(
                    'pytabkit'
                ),
            'model':
                'RealMLP_TD_Classifier',
            'official_tuned_default_recipe':
                OFFICIAL_REALMLP_TD_CLASS_DEFAULTS,
            'experiment_overrides':
                EXPERIMENT_OVERRIDES,
            'effective_realmlp_config_by_fold':
                effective_configs,
            'outer_cv': {
                'type':
                    'StratifiedKFold',
                'n_splits': 3,
                'shuffle': True,
                'random_state':
                    OUTER_SEED,
            },
            'outer_fold_seeds': {
                str(index): seed
                for index, seed
                in enumerate(
                    FOLD_SEEDS,
                    start=1,
                )
            },
            'baseline': {
                'name':
                    'GBDT_mean',
                'source':
                    str(
                        STAGE7_OOF_PATH
                        .relative_to(ROOT)
                    ),
                'retrained': False,
            },
            'protocol': {
                'device': 'cpu',
                'n_cv': 1,
                'n_refit': 0,
                'n_ens': 1,
                'hpo': False,
                'bagging_or_ensembling':
                    False,
                'calibration': False,
                'class_weighting':
                    False,
                'balancing_or_sampling':
                    False,
                'threshold_optimization':
                    False,
                'diagnostic_threshold':
                    THRESHOLD,
                'preprocessing':
                    (
                        'official RealMLP '
                        'native fold-local pipeline'
                    ),
            },
            'fold_metrics':
                fold_rows,
            'realmlp_oof_metrics':
                realmlp_metrics,
            'gbdt_mean_oof_metrics':
                baseline_metrics,
            'delta_realmlp_minus_gbdt_mean':
                deltas,
            'decision':
                decide(
                    deltas['Gini'],
                    fold_gini_deltas,
                ),
            'final_test_used':
                False,
            'runtime_seconds':
                total_runtime,
            'limitations': [
                (
                    'Random CV does not prove '
                    'temporal stability.'
                ),
                (
                    'Three folds are not a '
                    'statistical-significance claim.'
                ),
                (
                    'Precision/Recall/F1 at 0.5 '
                    'are diagnostic only.'
                ),
                (
                    'Final test was not used.'
                ),
                (
                    'One tuned-default RealMLP '
                    'recipe was evaluated without '
                    'KOMUS-specific HPO.'
                ),
            ],
        }

        summary_payload = {
            'experiment':
                'Stage 11 V1',
            'decision':
                result['decision'],
            'primary_metric':
                'OOF Gini',
            'realmlp_oof_gini':
                realmlp_metrics[
                    'Gini'
                ],
            'gbdt_mean_oof_gini':
                baseline_metrics[
                    'Gini'
                ],
            'delta_gini':
                deltas['Gini'],
            'fold_gini_deltas':
                fold_gini_deltas,
            'final_test_used':
                False,
            'result_artifacts': [
                str(
                    RESULT_PATH
                    .relative_to(ROOT)
                ),
                str(
                    OOF_PATH
                    .relative_to(ROOT)
                ),
            ],
        }

        # Важен порядок.
        # Completed RESULT_PATH создаётся последним.
        atomic_npz(
            OOF_PATH,
            working_indices=
                working_indices,
            target=
                y_working,
            fold=
                fold,
            gbdt_mean_probability=
                gbdt_mean,
            realmlp_oof_probability=
                realmlp_oof,
        )

        atomic_json(
            SUMMARY_PATH,
            summary_payload,
        )

        atomic_json(
            RESULT_PATH,
            result,
        )

        # Только после успешного сохранения
        # всего финального комплекта.
        CHECKPOINT_PATH.unlink(
            missing_ok=True
        )

        finished = True

        set_stage11_progress(
            fold_id=3,
            phase='эксперимент завершён',
            fold_started=None,
            last_delta_gini=
                deltas['Gini'],
        )

        render_stage11_panel(
            state,
            session_started,
            base_runtime,
        )

        show_stage11_result(
            result
        )

        return result

    except KeyboardInterrupt:
        if not finished:
            state['status'] = (
                'in_progress'
            )

            state['phase'] = (
                'остановлено пользователем'
            )

            state[
                'runtime_seconds'
            ] = (
                base_runtime
                + (
                    time.monotonic()
                    - session_started
                )
            )

            # Завершённые фолды сохраняются.
            # Незавершённый текущий fold
            # при resume будет выполнен заново.
            save_stage11_checkpoint(
                state
            )

            set_stage11_progress(
                phase=
                    'остановлено пользователем'
            )

            render_stage11_panel(
                state,
                session_started,
                base_runtime,
            )

        raise

    except Exception:
        if not finished:
            state['status'] = (
                'in_progress'
            )

            state['phase'] = (
                'ошибка; checkpoint сохранён'
            )

            state[
                'runtime_seconds'
            ] = (
                base_runtime
                + (
                    time.monotonic()
                    - session_started
                )
            )

            save_stage11_checkpoint(
                state
            )

            set_stage11_progress(
                phase=
                    'ошибка; checkpoint сохранён'
            )

            render_stage11_panel(
                state,
                session_started,
                base_runtime,
            )

        raise

    finally:
        stop_event.set()
        heartbeat.join(
            timeout=2.0
        )

In [5]:
stage11_result = run_stage11()

Внешний фолд,1/3
Текущая стадия,подготовка (fresh)
Завершено фолдов,0/3
Время текущего фолда,—
Общее время Stage 11,0 сек
Среднее время завершённого фолда,—
Осталось,появится после первого завершённого фолда
Checkpoint,создан; завершённых фолдов пока нет
Последний ΔGini,—


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=256` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
d:\Projects\komus-work\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, 

Stage 11 V1 завершён.
RealMLP Gini:   0.793325
GBDT_mean Gini: 0.806399
ΔGini:          -0.013074
Решение:        RealMLP хуже baseline (inferior)
Общее время:    2 ч 58 мин 50 сек


# Результат исследования

## ФАКТЫ

Заполнить по сохранённому JSON после будущего полного run: OOF metrics RealMLP, GBDT_mean, Δ и fold evidence.

## ИНТЕРПРЕТАЦИЯ

Использовать только заранее зафиксированное decision rule по OOF Gini; Recall/F1@0.5 не меняют classification.

## ОГРАНИЧЕНИЯ

Random CV не доказывает temporal stability; три folds не являются claim о statistical significance; final test не использован.

## СЛЕДУЮЩИЙ ШАГ

Результат передаётся Technical Coordinator для решения о завершении или продолжении model-only ветки.
